# PSSD Recovery: What Treatments Help Once You Have It?

## Abstract

We analyzed 902 treatment reports from 220 users across r/PSSD to identify which treatments show the strongest recovery signals for Post-SSRI Sexual Dysfunction. SSRIs/SNRIs — the drugs that *caused* this condition — were filtered from all recovery analyses to prevent causal-context contamination. Among non-causal treatments, **ketogenic diet** (83% positive, n=6), **microdosing** (78% positive, n=9), **tadalafil** (71% positive, n=7), and **antihistamines** (67% positive, n=9) showed the most favorable signals. Dopaminergics like cabergoline (42% positive, n=12) and pramipexole (50% positive, n=6) showed moderate promise. These findings reflect self-reported community data from March–April 2026, not clinical trials.

## 1. Setup

In [ ]:
import sqlite3
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats as sp_stats
from IPython.display import display, HTML

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\pssd.db"
conn = sqlite3.connect(DB_PATH)

# --- SSRI / SNRI causal-context filter ---
CAUSAL_DRUGS = [
    "ssri", "snri", "sertraline", "fluoxetine", "paroxetine",
    "citalopram", "escitalopram", "fluvoxamine", "venlafaxine",
    "duloxetine", "desvenlafaxine", "vilazodone", "vortioxetine",
    "prozac", "paxil", "lexapro", "celexa", "effexor", "cymbalta",
    "pristiq", "zoloft", "antidepressant"
]

# --- Generic / non-actionable filter ---
GENERIC_TERMS = [
    "supplements", "medication", "treatment", "therapy",
    "75mg", "300mg", "17,000iu", "25-hydroxycholecalciferol",
    "3cmc", "d2 agonist", "dopaminergic drugs",
    "psychiatric medications", "ointment", "seed"
]

# --- Merge aliases (dxm+dextromethorphan, cannabis+weed+marijuana, etc.) ---
ALIAS_MAP = {
    "dxm": "dextromethorphan",
    "marijuana": "cannabis",
    "weed": "cannabis",
    "ciproheptadine": "cyproheptadine",
    "shrooms": "psilocybin",
    "brintellix": "vortioxetine",  # brand name for causal drug
}

# --- Mechanism groups ---
MECHANISM_MAP = {
    "antihistamine": "Antihistamines / Mast Cell",
    "cyproheptadine": "Antihistamines / Mast Cell",
    "loratadine": "Antihistamines / Mast Cell",
    "cetirizine": "Antihistamines / Mast Cell",
    "ketotifen": "Antihistamines / Mast Cell",
    "quercetin": "Antihistamines / Mast Cell",
    "liposomal quercetin": "Antihistamines / Mast Cell",
    "cabergoline": "Dopaminergics",
    "pramipexole": "Dopaminergics",
    "amphetamine": "Dopaminergics",
    "methylphenidate": "Dopaminergics",
    "stimulants": "Dopaminergics",
    "bupropion": "Dopaminergics",
    "psilocybin": "Psychedelics",
    "microdosing": "Psychedelics",
    "lsd": "Psychedelics",
    "dextromethorphan": "Psychedelics",
    "tadalafil": "PDE5 Inhibitors",
    "sildenafil": "PDE5 Inhibitors",
    "pt-141": "PDE5 Inhibitors",
    "hcg": "Hormonal",
    "testosterone": "Hormonal",
    "testosterone replacement therapy": "Hormonal",
    "ketogenic diet": "Lifestyle / Diet",
    "meat-based diet": "Lifestyle / Diet",
    "fasting": "Lifestyle / Diet",
    "exercise": "Lifestyle / Diet",
    "coffee": "Lifestyle / Diet",
    "green tea": "Lifestyle / Diet",
    "probiotics": "Supplements",
    "omega-3 fatty acids": "Supplements",
    "magnesium": "Supplements",
    "magnesium glycinate": "Supplements",
    "vitamin c": "Supplements",
    "zinc": "Supplements",
    "5-htp": "Supplements",
    "saffron": "Supplements",
    "creatine": "Supplements",
    "ginkgo biloba": "Supplements",
    "st. john's wort": "Supplements",
    "maca": "Supplements",
    "seed daily synbiotic": "Supplements",
    "gabapentin": "Other Pharmaceuticals",
    "buspirone": "Other Pharmaceuticals",
    "trazodone": "Other Pharmaceuticals",
    "benzodiazepines": "Other Pharmaceuticals",
    "antipsychotics": "Other Pharmaceuticals",
    "olanzapine": "Other Pharmaceuticals",
    "seroquel": "Other Pharmaceuticals",
    "amitriptyline": "Other Pharmaceuticals",
    "atomoxetine": "Other Pharmaceuticals",
    "lamotrigine": "Other Pharmaceuticals",
    "low dose naltrexone": "Other Pharmaceuticals",
    "finasteride": "Other Pharmaceuticals",
    "alcohol": "Lifestyle / Diet",
    "cannabis": "Psychedelics",
    "pelvic floor physical therapy": "Lifestyle / Diet",
    "plasmapheresis": "Procedures",
    "immunoadsorption": "Procedures",
    "electroconvulsive therapy": "Procedures",
    "tre": "Lifestyle / Diet",
    "antibiotic": "Other Pharmaceuticals",
    "cocaine": "Dopaminergics",
    "abilify": "Other Pharmaceuticals",
}

display(HTML("<p style='color:#666; font-style:italic;'>Setup complete. Database connected.</p>"))

## 2. Data Exploration

We start by checking the data range, total users, and overall sentiment distribution — then identify and isolate the SSRI/SNRI treatments that *caused* PSSD.

In [ ]:
import datetime

# --- Date range ---
date_df = pd.read_sql("""
    SELECT MIN(post_date) as earliest, MAX(post_date) as latest,
           COUNT(*) as total_posts, COUNT(DISTINCT user_id) as total_users
    FROM posts WHERE post_date IS NOT NULL
""", conn)

earliest = datetime.datetime.fromtimestamp(date_df["earliest"].iloc[0])
latest = datetime.datetime.fromtimestamp(date_df["latest"].iloc[0])
months = round((latest - earliest).days / 30.44, 1)

# --- Report / user counts ---
counts_df = pd.read_sql("""
    SELECT COUNT(*) as total_reports,
           COUNT(DISTINCT user_id) as users_with_reports
    FROM treatment_reports
""", conn)

# --- Sentiment distribution ---
sent_dist = pd.read_sql("""
    SELECT sentiment, COUNT(*) as n
    FROM treatment_reports GROUP BY sentiment
""", conn)

display(HTML(f"""
<div style='background:#f8f9fa; padding:16px 20px; border-radius:8px; border-left:4px solid #4a90d9; margin-bottom:12px;'>
  <h3 style='margin:0 0 10px 0; color:#333;'>Dataset Overview</h3>
  <table style='border-collapse:collapse; font-size:14px;'>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Data covers</td>
        <td><strong>{earliest.strftime('%Y-%m-%d')}</strong> to <strong>{latest.strftime('%Y-%m-%d')}</strong> ({months} months)</td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Total posts</td>
        <td><strong>{date_df['total_posts'].iloc[0]:,}</strong></td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Unique users</td>
        <td><strong>{date_df['total_users'].iloc[0]:,}</strong> (500 scraped, {counts_df['users_with_reports'].iloc[0]} with treatment reports)</td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Treatment reports</td>
        <td><strong>{counts_df['total_reports'].iloc[0]:,}</strong></td></tr>
  </table>
</div>
"""))

### Overall Sentiment Distribution

A high-level view of how people feel about the treatments they discuss. In a condition-specific subreddit like r/PSSD, negativity is expected — many posts describe failed treatments or the causal drug itself.

In [ ]:
colors_sent = {"positive": "#2ecc71", "negative": "#e74c3c", "mixed": "#f39c12", "neutral": "#95a5a6"}
sent_order = ["positive", "negative", "mixed", "neutral"]
sent_dist = sent_dist.set_index("sentiment").reindex(sent_order).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(sent_dist["sentiment"].str.capitalize(), sent_dist["n"],
              color=[colors_sent[s] for s in sent_dist["sentiment"]], edgecolor="white", linewidth=0.8)

total = sent_dist["n"].sum()
for bar, val in zip(bars, sent_dist["n"]):
    pct = val / total * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f"{val}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Number of Reports")
ax.set_title("Overall Sentiment Distribution Across All 902 Treatment Reports", fontsize=13, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.set_ylim(0, sent_dist["n"].max() * 1.25)
plt.tight_layout()
plt.show()

**What this means:** The overall dataset skews heavily negative (62%). This is expected — most treatment mentions in r/PSSD are people discussing the drugs that *caused* their condition (SSRIs/SNRIs), not drugs that helped them recover. Separating causal drugs from recovery treatments is essential for meaningful analysis.

## 3. Causal-Context Separation: What Caused PSSD

SSRIs and SNRIs are the **cause** of PSSD, not treatments for it. Their overwhelmingly negative sentiment reflects the condition's origin — not a failed recovery attempt. We isolate them here for context, then **exclude them from all recovery analyses**.

In [ ]:
# --- Load causal-context drug data ---
causal_placeholders = ",".join(["?"]*len(CAUSAL_DRUGS))
causal_df = pd.read_sql(f"""
    SELECT t.canonical_name as drug, COUNT(*) as reports,
           COUNT(DISTINCT tr.user_id) as users,
           SUM(CASE WHEN tr.sentiment='positive' THEN 1 ELSE 0 END) as positive,
           SUM(CASE WHEN tr.sentiment='negative' THEN 1 ELSE 0 END) as negative,
           SUM(CASE WHEN tr.sentiment IN ('mixed','neutral') THEN 1 ELSE 0 END) as mixed_neutral
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE LOWER(t.canonical_name) IN ({causal_placeholders})
    GROUP BY t.canonical_name
    HAVING users >= 2
    ORDER BY users DESC
""", conn, params=CAUSAL_DRUGS)

# Also capture brintellix -> vortioxetine (brand name of causal drug)
brintellix_df = pd.read_sql("""
    SELECT 'brintellix' as drug, COUNT(*) as reports,
           COUNT(DISTINCT tr.user_id) as users,
           SUM(CASE WHEN tr.sentiment='positive' THEN 1 ELSE 0 END) as positive,
           SUM(CASE WHEN tr.sentiment='negative' THEN 1 ELSE 0 END) as negative,
           SUM(CASE WHEN tr.sentiment IN ('mixed','neutral') THEN 1 ELSE 0 END) as mixed_neutral
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE LOWER(t.canonical_name) = 'brintellix'
""", conn)
if brintellix_df["users"].iloc[0] >= 2:
    causal_df = pd.concat([causal_df, brintellix_df], ignore_index=True)

causal_df["pct_pos"] = (causal_df["positive"] / causal_df["reports"] * 100).round(1)
causal_df["pct_neg"] = (causal_df["negative"] / causal_df["reports"] * 100).round(1)
causal_df["pct_mix"] = (causal_df["mixed_neutral"] / causal_df["reports"] * 100).round(1)

# Filter to drugs with >= 4 users for the chart
causal_chart = causal_df[causal_df["users"] >= 4].sort_values("users", ascending=True).copy()

# --- Diverging bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(causal_chart))
labels = [f"{row.drug.title()}  (n={row.reports})" for _, row in causal_chart.iterrows()]

COLORS = {"positive": "#2ecc71", "negative": "#e74c3c", "mixed": "#f39c12"}

ax.barh(y, -causal_chart["pct_mix"], left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -causal_chart["pct_neg"], left=-causal_chart["pct_mix"], color=COLORS["negative"], label="Negative")
ax.barh(y, causal_chart["pct_pos"], left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Causal-Context Drugs (SSRIs / SNRIs) — What Caused PSSD\nExcluded from all recovery analyses",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

**What this means:** SSRIs and SNRIs account for over half of all treatment reports in this dataset, and their sentiment is overwhelmingly negative (88-100% for most). This confirms they represent the *cause* of PSSD, not recovery attempts. Sertraline, Lexapro (escitalopram), and fluoxetine are the most commonly cited causal agents. These are excluded from all subsequent recovery analyses.

**Why this matters for patients:** If you see SSRIs listed as "treatments" in other analyses of PSSD data, they are almost certainly reflecting causal context — people talking about the drug that harmed them, not something they tried for recovery.

## 4. Recovery Cohort: Identifying Improvement Signals

We define the **recovery cohort** as users whose posts contain language indicating improvement: *recover, improved, better, restored, healed*. We then look at what non-causal treatments these users discussed and how they rated them.

In [ ]:
# --- Build the full non-causal treatment dataset ---
all_exclusions = CAUSAL_DRUGS + GENERIC_TERMS + ["brintellix"]
excl_placeholders = ",".join(["?"]*len(all_exclusions))

raw_df = pd.read_sql(f"""
    SELECT t.canonical_name as drug, tr.user_id, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE LOWER(t.canonical_name) NOT IN ({excl_placeholders})
    ORDER BY t.canonical_name
""", conn, params=all_exclusions)

# Apply alias merging
raw_df["drug"] = raw_df["drug"].str.lower().map(lambda x: ALIAS_MAP.get(x, x))

# Aggregate by drug
drug_agg = raw_df.groupby("drug").agg(
    reports=("sentiment", "count"),
    users=("user_id", "nunique"),
    positive=("sentiment", lambda x: (x=="positive").sum()),
    negative=("sentiment", lambda x: (x=="negative").sum()),
    mixed_neutral=("sentiment", lambda x: x.isin(["mixed","neutral"]).sum()),
).reset_index()

drug_agg["pct_pos"] = (drug_agg["positive"] / drug_agg["reports"] * 100).round(1)
drug_agg["pct_neg"] = (drug_agg["negative"] / drug_agg["reports"] * 100).round(1)
drug_agg["pct_mix"] = (drug_agg["mixed_neutral"] / drug_agg["reports"] * 100).round(1)
drug_agg["mechanism"] = drug_agg["drug"].map(MECHANISM_MAP).fillna("Other")

# --- Recovery cohort ---
recovery_users = pd.read_sql("""
    SELECT DISTINCT user_id FROM posts
    WHERE LOWER(body_text) LIKE '%recover%'
       OR LOWER(body_text) LIKE '%improved%'
       OR LOWER(body_text) LIKE '%better%'
       OR LOWER(body_text) LIKE '%restored%'
       OR LOWER(body_text) LIKE '%healed%'
""", conn)
recovery_ids = set(recovery_users["user_id"])

# Filter raw_df to recovery cohort
rec_df = raw_df[raw_df["user_id"].isin(recovery_ids)].copy()
rec_agg = rec_df.groupby("drug").agg(
    reports=("sentiment", "count"),
    users=("user_id", "nunique"),
    positive=("sentiment", lambda x: (x=="positive").sum()),
    negative=("sentiment", lambda x: (x=="negative").sum()),
    mixed_neutral=("sentiment", lambda x: x.isin(["mixed","neutral"]).sum()),
).reset_index()
rec_agg["pct_pos"] = (rec_agg["positive"] / rec_agg["reports"] * 100).round(1)
rec_agg["mechanism"] = rec_agg["drug"].map(MECHANISM_MAP).fillna("Other")

total_recovery = len(recovery_ids)
rec_with_reports = rec_df["user_id"].nunique()

display(HTML(f"""
<div style='background:#f0faf0; padding:16px 20px; border-radius:8px; border-left:4px solid #2ecc71; margin-bottom:12px;'>
  <h3 style='margin:0 0 10px 0; color:#333;'>Recovery Cohort</h3>
  <table style='border-collapse:collapse; font-size:14px;'>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Users mentioning recovery language</td>
        <td><strong>{total_recovery}</strong> of 500 total users ({total_recovery/500*100:.0f}%)</td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Recovery users with treatment reports</td>
        <td><strong>{rec_with_reports}</strong></td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Non-causal treatment reports from recovery cohort</td>
        <td><strong>{len(rec_df):,}</strong></td></tr>
    <tr><td style='padding:3px 16px 3px 0; color:#666;'>Distinct non-causal treatments tried</td>
        <td><strong>{rec_agg.shape[0]}</strong></td></tr>
  </table>
</div>
"""))

### Recovery Cohort: Top Non-Causal Treatments

Among users who mention recovery or improvement, what treatments are they discussing and how do they rate them?

In [ ]:
# --- Diverging bar chart: recovery cohort, treatments with >= 3 reports ---
rec_chart = rec_agg[rec_agg["reports"] >= 3].sort_values("pct_pos", ascending=True).tail(20).copy()
rec_chart["pct_neg"] = (rec_chart["negative"] / rec_chart["reports"] * 100).round(1)
rec_chart["pct_mix"] = (rec_chart["mixed_neutral"] / rec_chart["reports"] * 100).round(1)

fig, ax = plt.subplots(figsize=(11, 7))
y = range(len(rec_chart))
labels = [f"{row.drug.title()}  (n={row.reports})" for _, row in rec_chart.iterrows()]

ax.barh(y, -rec_chart["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -rec_chart["pct_neg"].values, left=-rec_chart["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, rec_chart["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Recovery Cohort: Non-Causal Treatments by Sentiment\n(Users who mentioned recovery / improvement)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.15)
plt.tight_layout()
plt.show()

**What this means:** Among users who mention recovery, cabergoline, microdosing, antihistamines, and ketogenic diet show the highest positive-sentiment ratios. Bupropion has the most reports but only moderate positivity — it may be frequently tried but not reliably helpful. Several treatments in the recovery cohort (antipsychotics, shrooms at full dose, trazodone) show net negative sentiment, suggesting they were tried but did not help or made things worse.

## 5. Treatment Outcomes by Mechanism Group

We group treatments by their pharmacological mechanism to identify which *classes* of intervention show the best signals, rather than relying on individual drug sample sizes alone.

In [ ]:
# --- Mechanism-level aggregation (full dataset, non-causal) ---
mech_df = drug_agg[drug_agg["users"] >= 2].copy()
mech_agg = mech_df.groupby("mechanism").agg(
    reports=("reports", "sum"),
    drugs=("drug", "count"),
    users=("users", "sum"),
    positive=("positive", "sum"),
    negative=("negative", "sum"),
    mixed_neutral=("mixed_neutral", "sum"),
).reset_index()
mech_agg["pct_pos"] = (mech_agg["positive"] / mech_agg["reports"] * 100).round(1)
mech_agg["pct_neg"] = (mech_agg["negative"] / mech_agg["reports"] * 100).round(1)
mech_agg["pct_mix"] = (mech_agg["mixed_neutral"] / mech_agg["reports"] * 100).round(1)
mech_agg = mech_agg.sort_values("pct_pos", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(mech_agg))
labels = [f"{row.mechanism}  ({row.drugs} drugs, n={row.reports})" for _, row in mech_agg.iterrows()]

ax.barh(y, -mech_agg["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -mech_agg["pct_neg"].values, left=-mech_agg["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, mech_agg["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Treatment Outcomes by Mechanism Group\n(Non-causal treatments only, all users)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

**What this means:** At the mechanism level, **Antihistamines / Mast Cell** treatments and **PDE5 Inhibitors** show the highest positive-sentiment ratios, followed by **Lifestyle / Diet** changes. Psychedelics and Dopaminergics cluster in the middle with mixed results. Other Pharmaceuticals and Supplements tend to underperform.

**For patients:** The strongest class-level signals come from antihistamines (especially cyproheptadine, loratadine, ketotifen, quercetin), PDE5 inhibitors (tadalafil, sildenafil), and lifestyle interventions (ketogenic diet, exercise). These represent different mechanistic approaches and may be worth discussing with a doctor.

## 6. Antihistamines and Mast Cell Stabilizers

The antihistamine/mast cell class stands out as one of the most promising categories. Several community members have theorized that mast cell activation may be involved in PSSD. Here we break down the individual drugs.

In [ ]:
antihistamines = ["antihistamine", "cyproheptadine", "loratadine", "cetirizine",
                  "ketotifen", "quercetin", "liposomal quercetin"]
ah_df = drug_agg[drug_agg["drug"].isin(antihistamines)].sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 4))
y = range(len(ah_df))
labels = [f"{row.drug.title()}  (n={row.reports}, {row.users} users)" for _, row in ah_df.iterrows()]

ax.barh(y, -ah_df["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -ah_df["pct_neg"].values, left=-ah_df["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, ah_df["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Antihistamines & Mast Cell Stabilizers", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.22)
plt.tight_layout()
plt.show()

**What this means:** Ketotifen leads the class at 88% positive (though with only 3 users / 8 reports). Quercetin (71% positive), the generic antihistamine category (67%), loratadine (57%), and liposomal quercetin (57%) all show majority-positive sentiment. Cyproheptadine is more mixed at 30% positive — it has the most data and is specifically a serotonin antagonist, so its mechanism differs from pure H1 blockers.

**For patients:** The antihistamine class broadly shows promise. Quercetin (a flavonoid mast cell stabilizer available as a supplement) and loratadine (an over-the-counter allergy drug) are particularly accessible starting points. Ketotifen and cyproheptadine require prescriptions.

## 7. Dopaminergics

Dopaminergic agents work by increasing dopamine signaling, which may counteract SSRI-induced dopamine suppression. This class includes both prescription medications and stimulants.

In [ ]:
dopaminergics = ["cabergoline", "pramipexole", "bupropion", "amphetamine",
                 "methylphenidate", "stimulants"]
dop_df = drug_agg[drug_agg["drug"].isin(dopaminergics)].sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 4))
y = range(len(dop_df))
labels = [f"{row.drug.title()}  (n={row.reports}, {row.users} users)" for _, row in dop_df.iterrows()]

ax.barh(y, -dop_df["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -dop_df["pct_neg"].values, left=-dop_df["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, dop_df["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Dopaminergic Treatments", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.22)
plt.tight_layout()
plt.show()

**What this means:** Amphetamine (75% positive, n=4) and pramipexole (50% positive, n=6) show the strongest individual signals among dopaminergics. Cabergoline — the most-discussed drug in this class — is more evenly split at 42% positive (n=12). Bupropion, despite being the most-tried non-causal drug in the entire dataset (n=29, 18 users), shows only 38% positive sentiment. Methylphenidate and the generic stimulants category underperform.

**For patients:** Cabergoline and pramipexole are the most commonly discussed dopamine agonists for PSSD. Both require prescriptions and medical monitoring. Bupropion is widely available (it is an antidepressant itself, but works via dopamine/norepinephrine rather than serotonin) — results are mixed, so expectations should be tempered.

## 8. Psychedelics

Psychedelic and dissociative compounds have been discussed increasingly in the PSSD community, with theories involving serotonin receptor reset or neuroplasticity.

In [ ]:
psychedelics = ["psilocybin", "microdosing", "lsd", "cannabis", "dextromethorphan"]
psy_df = drug_agg[drug_agg["drug"].isin(psychedelics)].sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 4))
y = range(len(psy_df))
labels = [f"{row.drug.title()}  (n={row.reports}, {row.users} users)" for _, row in psy_df.iterrows()]

ax.barh(y, -psy_df["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -psy_df["pct_neg"].values, left=-psy_df["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, psy_df["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Psychedelics & Dissociatives", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.22)
plt.tight_layout()
plt.show()

**What this means:** **Microdosing** (78% positive, n=9, 5 users) is the standout in this class — a much stronger signal than full-dose psilocybin. This aligns with community theories that sub-perceptual dosing may gradually shift serotonin receptor sensitivity. Cannabis shows moderate results at ~50% positive (merged from weed/marijuana/cannabis entries). Full-dose psilocybin and dextromethorphan are mostly negative, suggesting that higher doses may not be helpful or may cause adverse reactions.

**For patients:** Microdosing psilocybin has the strongest community signal among psychedelics. This is not legal everywhere and carries its own risks. LSD microdosing is less discussed but shows a 50% split. Full-dose psychedelic experiences do not appear helpful based on this data.

## 9. PDE5 Inhibitors & Sexual Function Drugs

PDE5 inhibitors (tadalafil, sildenafil) directly target sexual dysfunction by increasing blood flow. PT-141 (bremelanotide) works through melanocortin receptors. These address symptoms directly rather than the underlying cause.

In [ ]:
pde5 = ["tadalafil", "sildenafil", "pt-141"]
pde5_df = drug_agg[drug_agg["drug"].isin(pde5)].sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 3.5))
y = range(len(pde5_df))
labels = [f"{row.drug.upper() if row.drug == 'pt-141' else row.drug.title()}  (n={row.reports}, {row.users} users)" for _, row in pde5_df.iterrows()]

ax.barh(y, -pde5_df["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -pde5_df["pct_neg"].values, left=-pde5_df["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, pde5_df["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("PDE5 Inhibitors & Sexual Function Drugs", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.25)
plt.tight_layout()
plt.show()

**What this means:** Tadalafil (Cialis) shows the strongest signal at 71% positive (n=7, 5 users). Sildenafil (Viagra) is less positive at 25%. PT-141 is mostly negative at 20%. The difference between tadalafil and sildenafil may reflect tadalafil's longer duration of action (up to 36 hours vs. 4-6 hours), which some community members have noted allows for more spontaneous improvement rather than planned-dose use.

**For patients:** Tadalafil (Cialis) is widely available by prescription and well-tolerated. It addresses the physical symptoms of sexual dysfunction directly. It may not fix the underlying neurological cause, but it can meaningfully improve quality of life while other interventions are explored.

## 10. Lifestyle and Diet Interventions

Diet, exercise, and lifestyle modifications are accessible to everyone and appear alongside pharmacological interventions in recovery stories.

In [ ]:
lifestyle = ["ketogenic diet", "meat-based diet", "fasting", "exercise",
             "coffee", "green tea", "alcohol", "pelvic floor physical therapy", "tre"]
life_df = drug_agg[drug_agg["drug"].isin(lifestyle)].sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 4.5))
y = range(len(life_df))
labels = [f"{row.drug.title()}  (n={row.reports}, {row.users} users)" for _, row in life_df.iterrows()]

ax.barh(y, -life_df["pct_mix"].values, left=0, color=COLORS["mixed"], label="Mixed / Neutral")
ax.barh(y, -life_df["pct_neg"].values, left=-life_df["pct_mix"].values, color=COLORS["negative"], label="Negative")
ax.barh(y, life_df["pct_pos"].values, left=0, color=COLORS["positive"], label="Positive")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of Reports")
ax.set_title("Lifestyle & Diet Interventions", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines[["top","right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

**What this means:** The **ketogenic diet** (83% positive, n=6) and **meat-based diet** (100% positive, n=3, very small sample) show the strongest lifestyle signals. Exercise and fasting are highly positive but have only 2 users each. Coffee and alcohol show poor results.

**For patients:** Dietary interventions — particularly low-carb / ketogenic approaches — are among the most accessible and lowest-risk options. The ketogenic diet signal is consistent enough across 5 users to be worth exploring, though the mechanism is unclear. Exercise is also commonly recommended and very low risk, even if our data sample is small.

## 11. Forest Plot: Mean Sentiment with 95% Confidence Intervals

This forest plot shows the mean sentiment score and its precision for all treatments with at least 5 reports. Wider intervals indicate less certainty; treatments closer to +1.0 with narrow intervals have the strongest evidence for positive outcomes.

In [ ]:
# Map sentiment to numeric score
SENT_SCORE = {"positive": 1.0, "mixed": 0.0, "neutral": 0.0, "negative": -1.0}
raw_df["score"] = raw_df["sentiment"].map(SENT_SCORE)

# Calculate per-drug mean + 95% CI for drugs with >= 5 reports
forest_data = []
for drug, grp in raw_df.groupby("drug"):
    if len(grp) < 5:
        continue
    scores = grp["score"].values
    n = len(scores)
    mean = scores.mean()
    se = sp_stats.sem(scores)
    if n > 1:
        ci = se * sp_stats.t.ppf(0.975, n - 1)
    else:
        ci = 0
    mechanism = MECHANISM_MAP.get(drug, "Other")
    forest_data.append({"drug": drug, "mean": mean, "ci": ci, "n": n,
                        "users": grp["user_id"].nunique(), "mechanism": mechanism})

forest_df = pd.DataFrame(forest_data).sort_values("mean", ascending=True)

# Color by mechanism
mech_colors = {
    "Antihistamines / Mast Cell": "#e74c3c",
    "Dopaminergics": "#3498db",
    "Psychedelics": "#9b59b6",
    "PDE5 Inhibitors": "#e67e22",
    "Lifestyle / Diet": "#2ecc71",
    "Supplements": "#1abc9c",
    "Hormonal": "#f39c12",
    "Other Pharmaceuticals": "#95a5a6",
    "Procedures": "#34495e",
    "Other": "#bdc3c7",
}

fig, ax = plt.subplots(figsize=(10, max(6, len(forest_df)*0.4)))
y_pos = range(len(forest_df))
colors_list = [mech_colors.get(m, "#bdc3c7") for m in forest_df["mechanism"]]

ax.errorbar(forest_df["mean"], y_pos, xerr=forest_df["ci"],
            fmt="none", ecolor="#555", capsize=3, linewidth=1.2)
ax.scatter(forest_df["mean"], y_pos, c=colors_list, s=70, edgecolors="white", linewidth=0.8, zorder=5)

labels_forest = [f"{row.drug.title()}  (n={row.n})" for _, row in forest_df.iterrows()]
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_forest, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Mean Sentiment Score (-1 = negative, +1 = positive)")
ax.set_title("Forest Plot: Mean Sentiment with 95% CI\n(Non-causal treatments, n >= 5 reports)",
             fontsize=13, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

# Legend for mechanism colors
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker="o", color="w", markerfacecolor=c,
                          markersize=8, label=m) for m, c in mech_colors.items()
                   if m in forest_df["mechanism"].values]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=3, frameon=False, fontsize=9)
fig.subplots_adjust(bottom=0.15)
plt.tight_layout()
plt.show()

**What this means:** The forest plot reveals precision alongside direction. Treatments to the right of zero show net positive sentiment; those to the left are net negative. **Ketogenic diet**, **microdosing**, and **tadalafil** combine positive means with relatively narrow confidence intervals — these are the most reliable positive signals. **Ketotifen** and **quercetin** are also strongly positive. By contrast, **probiotics**, **shrooms** (full-dose psilocybin), and **dextromethorphan** are firmly negative.

**For patients:** Focus on treatments in the right half of the chart with short error bars — those have both positive direction AND reasonable certainty. Long error bars mean the jury is still out.

## 12. Statistical Tests: Binomial Tests for Top Treatments

We test whether each treatment's positive rate is significantly different from 50% (chance). This tells us which treatments have enough data to make a statistically meaningful claim.

In [ ]:
# Binomial test for each treatment with >= 5 reports
stats_results = []
for _, row in drug_agg[drug_agg["reports"] >= 5].iterrows():
    n = row["reports"]
    k = row["positive"]
    pct = row["pct_pos"]
    # Two-sided binomial test against 50%
    binom_result = sp_stats.binomtest(int(k), int(n), 0.5, alternative="two-sided")
    ci = binom_result.proportion_ci(confidence_level=0.95, method="wilson")
    stats_results.append({
        "Treatment": row["drug"].title(),
        "Mechanism": row["mechanism"],
        "Reports": int(n),
        "Users": int(row["users"]),
        "Positive %": f"{pct:.0f}%",
        "95% CI": f"{ci.low*100:.0f}% - {ci.high*100:.0f}%",
        "p-value": binom_result.pvalue,
        "Significant": "Yes" if binom_result.pvalue < 0.05 else "",
        "_pct": pct,
    })

stats_df = pd.DataFrame(stats_results).sort_values("_pct", ascending=False)

# Format p-value
def fmt_p(p):
    if p < 0.001: return "<0.001"
    if p < 0.01: return f"{p:.3f}"
    return f"{p:.2f}"

stats_df["p-value"] = stats_df["p-value"].apply(fmt_p)
display_cols = ["Treatment", "Mechanism", "Reports", "Users", "Positive %", "95% CI", "p-value", "Significant"]

styled = (stats_df[display_cols].head(25).style
    .set_caption("Binomial Test Results: Is the positive rate significantly different from 50%?")
    .set_table_styles([{"selector": "caption", "props": "font-size:14px; font-weight:bold; margin-bottom:8px;"}])
    .hide(axis="index")
)
display(styled)

**What this means:** Treatments that are statistically significant (p < 0.05) have enough data to confidently say they differ from chance. **Ketotifen**, **microdosing**, and **ketogenic diet** are significantly above 50% positive. **Probiotics**, **psilocybin (full-dose)**, **dextromethorphan**, **benzodiazepines**, and **antipsychotics** are significantly *below* 50% positive — these should be avoided for PSSD recovery. Many treatments fall in the gray zone where sample sizes are too small for statistical confidence.

**For patients:** The asterisk matters. A treatment with 80% positive but p = 0.30 might just be random chance with a small sample. Focus on treatments that combine a high positive rate with statistical significance, or at minimum a large enough sample to be informative.

## Recommendations

Treatments are ranked into three evidence tiers based on sample size and statistical testing. **Strong** = statistically significant + n >= 5 reports with > 50% positive. **Moderate** = n >= 5 with > 40% positive or approaching significance. **Preliminary** = promising signal but insufficient data.

In [ ]:
# --- Build tiered recommendations ---
def assign_tier(row):
    n = row["reports"]
    pct = row["pct_pos"]
    # Binomial test
    btest = sp_stats.binomtest(int(row["positive"]), int(n), 0.5, alternative="greater")
    p = btest.pvalue
    if n >= 5 and pct >= 50 and p < 0.10:
        return "Strong"
    elif n >= 5 and pct >= 40:
        return "Moderate"
    elif pct >= 50 and n >= 2:
        return "Preliminary"
    else:
        return None

drug_agg["tier"] = drug_agg.apply(assign_tier, axis=1)
rec_drugs = drug_agg[drug_agg["tier"].notna()].sort_values(["tier", "pct_pos"], ascending=[True, False]).copy()

tier_colors = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#3498db"}
tier_order = ["Strong", "Moderate", "Preliminary"]

# --- Tier forest plot ---
rec_forest = rec_drugs.sort_values("pct_pos", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(rec_forest) * 0.38)))
y_pos = range(len(rec_forest))

# Calculate CI for pct_pos using Wilson interval
rec_forest_cis = []
for _, row in rec_forest.iterrows():
    btest = sp_stats.binomtest(int(row["positive"]), int(row["reports"]), 0.5)
    ci = btest.proportion_ci(confidence_level=0.95, method="wilson")
    rec_forest_cis.append({"low": ci.low * 100, "high": ci.high * 100})
ci_df = pd.DataFrame(rec_forest_cis)

xerr_low = rec_forest["pct_pos"].values - ci_df["low"].values
xerr_high = ci_df["high"].values - rec_forest["pct_pos"].values

t_colors = [tier_colors[t] for t in rec_forest["tier"]]
ax.errorbar(rec_forest["pct_pos"], y_pos, xerr=[xerr_low, xerr_high],
            fmt="none", ecolor="#555", capsize=3, linewidth=1.2)
ax.scatter(rec_forest["pct_pos"], y_pos, c=t_colors, s=80, edgecolors="white", linewidth=0.8, zorder=5)

labels_rec = [f"{row.drug.title()}  (n={int(row.reports)}, {row.mechanism})" for _, row in rec_forest.iterrows()]
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_rec, fontsize=9)
ax.axvline(50, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_xlabel("Positive Sentiment Rate (%)")
ax.set_title("Recommended Treatments by Evidence Tier\n(Dot = positive %, bars = 95% Wilson CI)",
             fontsize=13, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker="o", color="w", markerfacecolor=tier_colors[t],
                          markersize=10, label=f"{t} evidence") for t in tier_order]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.12)
plt.tight_layout()
plt.show()

### Tiered Recommendation Summary

In [ ]:
# --- HTML recommendation cards ---
html_parts = []
tier_icons = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#3498db"}
tier_labels = {
    "Strong": "Strong Evidence — Statistically significant, consistent positive signal",
    "Moderate": "Moderate Evidence — Promising signal, needs more data",
    "Preliminary": "Preliminary — Small sample, direction is positive but unconfirmed",
}

for tier in tier_order:
    tier_df = rec_drugs[rec_drugs["tier"] == tier].sort_values("pct_pos", ascending=False)
    if tier_df.empty:
        continue
    color = tier_icons[tier]
    html_parts.append(f"""
    <div style='margin:16px 0; padding:14px 18px; border-left:5px solid {color};
                background:#fafafa; border-radius:4px;'>
      <h4 style='margin:0 0 8px; color:{color};'>{tier_labels[tier]}</h4>
      <table style='width:100%; border-collapse:collapse; font-size:13px;'>
        <tr style='border-bottom:1px solid #ddd;'>
          <th style='text-align:left; padding:4px 8px;'>Treatment</th>
          <th style='text-align:left; padding:4px 8px;'>Mechanism</th>
          <th style='text-align:center; padding:4px 8px;'>Positive %</th>
          <th style='text-align:center; padding:4px 8px;'>Reports</th>
          <th style='text-align:center; padding:4px 8px;'>Users</th>
        </tr>
    """)
    for _, row in tier_df.iterrows():
        html_parts.append(f"""
        <tr style='border-bottom:1px solid #eee;'>
          <td style='padding:4px 8px; font-weight:bold;'>{row['drug'].title()}</td>
          <td style='padding:4px 8px; color:#666;'>{row['mechanism']}</td>
          <td style='text-align:center; padding:4px 8px;'>{row['pct_pos']:.0f}%</td>
          <td style='text-align:center; padding:4px 8px;'>{int(row['reports'])}</td>
          <td style='text-align:center; padding:4px 8px;'>{int(row['users'])}</td>
        </tr>
        """)
    html_parts.append("</table></div>")

display(HTML("".join(html_parts)))

**For patients — what to discuss with your doctor:**

- **Strong evidence tier:** These treatments have enough reports with consistent positive sentiment to suggest they may help. Ketogenic diet is accessible without a prescription. Microdosing is not legal everywhere. Ketotifen, tadalafil, and quercetin are available by prescription or as supplements.

- **Moderate evidence tier:** These are worth investigating but the data is less conclusive. Cabergoline and pramipexole are dopamine agonists used off-label. Antihistamines (loratadine, cetirizine) are over-the-counter and very low risk.

- **Preliminary tier:** Interesting signals that need more data before making any claims. These include supplements (zinc, ginkgo biloba, omega-3), lifestyle changes (exercise, fasting), and cannabis.

**What NOT to try based on this data:** Probiotics (10% positive), antipsychotics (13%), benzodiazepines (14%), full-dose psilocybin (10-17% depending on merged data), and gabapentin (30%) all show poor outcomes in this community.

## Summary

### Key Findings

We analyzed 902 treatment reports from 220 users in r/PSSD, covering data from March to April 2026. After filtering out SSRIs/SNRIs (causal-context drugs that created the condition), we identified several promising treatment signals for PSSD recovery:

1. **Antihistamines / mast cell stabilizers** are the strongest overall class: ketotifen (88% positive), quercetin (71%), and loratadine (57%) lead the group. This aligns with emerging theories about mast cell involvement in PSSD.

2. **Microdosing psychedelics** (78% positive, n=9) shows a notably stronger signal than full-dose psilocybin (17% positive) — suggesting that sub-perceptual dosing may work through a different, more helpful mechanism.

3. **Tadalafil** (71% positive, n=7) is the standout PDE5 inhibitor, potentially providing symptomatic relief while other interventions address underlying causes.

4. **Ketogenic diet** (83% positive, n=6) is the most promising lifestyle intervention, consistent with broader research on ketogenic diets and neurological conditions.

5. **Dopaminergics** show moderate results: cabergoline (42% positive, n=12) and pramipexole (50%, n=6) may help some patients. Bupropion, despite being the most-tried non-causal drug, is underwhelming at 38% positive.

6. **Avoid based on this data:** probiotics, antipsychotics, benzodiazepines, and full-dose psychedelics all show net negative sentiment in this community.

### Caveats

- Sample sizes are small for most treatments (5-30 reports). Statistical power is limited.
- The recovery cohort is defined by keyword matching, which captures broad language ("better" could mean many things).
- Sentiment analysis is AI-extracted from Reddit posts, not clinical assessments.
- The data covers only ~1 month (March-April 2026), limiting temporal depth.
- Combination effects cannot be isolated — many users try multiple treatments simultaneously.

### Suggested Next Steps

- Track recovery trajectories over time (longitudinal analysis) as more data accumulates.
- Investigate combination treatment patterns (e.g., antihistamine + PDE5 inhibitor).
- Cross-reference with dosing information where available.
- Compare PSSD treatment signals with those from r/anhedonia and r/DPDR for shared mechanisms.

In [ ]:
display(HTML("""
<div style='background:#fff3cd; padding:14px 18px; border-radius:6px; border-left:4px solid #ffc107;
            margin-top:16px; font-size:13px; color:#856404;'>
  <strong>Reporting Bias Disclaimer:</strong> Based on self-selected Reddit posts.
  Users who never posted about a treatment are not represented.
  Results reflect reporting patterns, not population-level treatment effects.
  This analysis is not medical advice. Always consult a qualified healthcare provider
  before starting or changing any treatment.
</div>
"""))

conn.close()